# BYOL Downstream Classification Evaluation — LoTSS initial_pure

Evaluates BYOL encoder representations on the LoTSS initial_pure 5-class task,
using the train/test split produced by the BYOL training run.

All **configurations** (`CONFIG`, `GRID_CONFIG`, `CM_CONFIG`) are defined in Section 0.
All **imports** are collected in Section 1.

Sections:
- **0** Configuration (`CONFIG`, `GRID_CONFIG`, `CM_CONFIG`)
- **1** Imports & Setup
- **2** Load Encoder
- **3** Load Projections (from BYOL run)
- **4** Classifier Utilities (`run_linear_probe`, `run_classifier_suite`)
- **5** Classifier Suite — load pre-computed or train on site (LR / KNN / RF / MLP)
- **6** Supervised Baseline Comparison
- **6b** Fine-tuning Comparison (per-class metrics across freeze strategies)
- **6c** F1 Comparison: BYOL vs Supervised (summary plot)
- **7** Hyperparameter Sweep Grid
- **8** Confusion Matrix (BYOL vs Supervised Baseline)
- **9** Label-Fraction Experiment

## 0. Configuration

In [1]:
CONFIG = {
    # ── Main BYOL run (Sections 2–5, 8–9) ─────────────────────────────────────
    "checkpoint":    "../outputs/byol_runs/enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar2_cov0.1_gamma0.25_f1_sw0.02_augextended_20260707_0934/byol_model_best.pt",
    "catalogue":     "../catalogues/lotss_initial_all.yaml",
    "colour_by":     "dataset",   # "dataset" | "label"
    "root":          "..",

    # ── Classification evaluation ──────────────────────────────────────────────
    "label_set":     "initial_pure",

    # ── Supervised baselines (Section 6) ──────────────────────────────────────
    "baselines_dir": "../outputs/supervised_baseline_classifiers",

    # ── Fine-tuning runs (Sections 6b, 6c) ────────────────────────────────────
    "finetuning_runs": [
        "../outputs/run_run_mlp_sw0_fl1_20260608_1632/",
        "../outputs/run_run_mlp_sw0.5_fl0.5_20260608_1713/",
        "../outputs/run_run_mlp_sw1_fl1_20260608_1600/",
        "../outputs/run_run_mlp_sw1_fl0.5_20260608_1634/",
    ],
    "finetuning_types": [          # all types compared in Section 6b
        "finetuning_freeze0",
        "finetuning_freeze4",
        "finetuning_supervised1",
    ],
    "finetuning_type": "finetuning_freeze0",   # single type used in Section 6c

    # ── BYOL runs root (Section 6c) ───────────────────────────────────────────
    "byol_runs_dir": "../outputs/byol_runs",
}

# ── Hyperparameter sweep (Section 7) ──────────────────────────────────────────
GRID_CONFIG = {
    "axis_x":        "f",            # label fraction  (f0, f0.1, f0.5, f1 …)
    "axis_y":        "sw",           # supervision wt  (sw0.01, sw0.1, sw1 …)
    "classifier":    "rf",           # rf | knn | lr
    "label_set":     "initial_pure",
    "feature_type":  "projections",
    "outputs_root":  "../outputs",
    "run_glob":      "enb0_*",
}

# ── Confusion matrix (Section 8) ──────────────────────────────────────────────
CM_CONFIG = {
    "label_set":       "initial_pure",
    "byol_classifier": "knn",            # "random_forest" | "knn" | "linear_probe"
    "baseline_model":  "enb0",           # substring matched against baseline run dir names
    "baseline_dir":    "../outputs/supervised_baseline_classifiers",
    "n_estimators":    200,
    "n_neighbors":     15,
}

OUT_DIR = "../outputs/figures/classification"
SEED    = 42

## 1. Imports & Setup

All third-party imports and project-local imports (`load_encoder`, `ALL_CLASS_NAMES`, `LABEL_SETS`).
`OUT_DIR` is created here if it does not exist.

In [2]:
import os, sys, re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix

# Add project src and scripts to path
_root = os.path.abspath("..")
for _p in [os.path.join(_root, "src"), os.path.join(_root, "scripts")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from embed_and_umap import load_encoder
from train_byol_classifiers import ALL_CLASS_NAMES, LABEL_SETS

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports OK")

Imports OK


## 2. Load Encoder

Loads the BYOL checkpoint and returns the online encoder + projector. Auto-detects model type from the checkpoint config.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

encoder, projector = load_encoder(CONFIG["checkpoint"], device)
encoder.eval()
projector.eval()

# Detect dimensions via dummy forward pass
with torch.no_grad():
    _dummy   = torch.zeros(1, 1, 89, 89).to(device)
    enc_dim  = encoder(_dummy).shape[-1]
    proj_dim = projector(encoder(_dummy)).shape[-1]

print(f"\nEncoder output dim  : {enc_dim}")
print(f"Projector output dim: {proj_dim}")


Device: cuda
Loading checkpoint: ../outputs/byol_runs/enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar2_cov0.1_gamma0.25_f1_sw0.02_augextended_20260707_0934/byol_model_best.pt
  Model type: efficientnet-b0
  Feature compression: mlp
  Projector output dim: 128

Encoder output dim  : 1280
Projector output dim: 128


## 3. Load Projections (from BYOL run)

Loads pre-extracted projections directly from the BYOL run's `data/` directory.
This ensures the train/test split is identical to what `train_byol_classifiers.py` and the
Protege pipeline use.

`projs_cache["lotss_initial_pure"]` contains:
- `projs`     — shape `(N, proj_dim)`
- `labels`    — shape `(N,)` int64, argmax over 5 initial classes
- `split_ids` — shape `(N,)` int64, 0=train / 2=test (no val in BYOL split)


In [4]:
projs_cache = {}

_run_dir  = Path(CONFIG["checkpoint"]).parent
_data_dir = _run_dir / "data"
_byol_dir = _data_dir / "byol"

# Derive shared splits directory from the checkpoint's data_seed
_ckpt_raw   = torch.load(CONFIG["checkpoint"], map_location="cpu", weights_only=False)
_data_seed  = int(_ckpt_raw["config"]["data_seed"])
_splits_dir = _run_dir.parent / "data_splits" / str(_data_seed)

def _load_label(name):
    """Load a label .npy from splits_dir (new runs) or data_dir (old runs)."""
    p = _splits_dir / name
    return np.load(p if p.exists() else _data_dir / name)

_X_train = np.load(_byol_dir / "labelled_train_projections.npy").astype(np.float32)
_X_test  = np.load(_byol_dir / "test_projections.npy").astype(np.float32)

_lab_labels_path = _splits_dir / "labelled_train_labels.npy"
if not _lab_labels_path.exists():
    _lab_labels_path = _data_dir / "labelled_train_labels.npy"
if _lab_labels_path.exists():
    _y_train_20 = np.load(_lab_labels_path)
    if len(_y_train_20) != len(_X_train):
        # labelled_train_labels.npy was written by a different f_label run.
        # Reconstruct by concatenating labelled + unlabelled labels (covers all of
        # train_idx in order, which matches labelled_train_projections.npy for f=1).
        _unlab_lbl = _splits_dir / "unlabelled_train_labels.npy"
        if not _unlab_lbl.exists():
            _unlab_lbl = _data_dir / "unlabelled_train_labels.npy"
        if _unlab_lbl.exists():
            _y_train_20 = np.concatenate([_y_train_20, np.load(_unlab_lbl)])
        if len(_y_train_20) != len(_X_train):
            raise ValueError(f"Label count {len(_y_train_20)} != projection count {len(_X_train)}")
else:
    _all_train = _load_label("train_labels.npy")
    _lab_idx   = _load_label("labelled_train_idx.npy")
    _y_train_20 = _all_train if len(_all_train) == len(_X_train) else _all_train[_lab_idx]
_y_test_20 = _load_label("test_labels.npy")

def _initial_pure_mask(y20):
    """Rows with exactly one label among the 5 initial classes (FRI/FRII/Hybrid/Spiral/Relaxed)."""
    return y20[:, 0:5].sum(axis=1) == 1

_tr_mask = _initial_pure_mask(_y_train_20)
_te_mask = _initial_pure_mask(_y_test_20)

_X_tr_cp = _X_train[_tr_mask]
_y_tr_cp = _y_train_20[_tr_mask][:, 0:5].argmax(axis=1).astype(np.int64)
_X_te_cp = _X_test[_te_mask]
_y_te_cp = _y_test_20[_te_mask][:, 0:5].argmax(axis=1).astype(np.int64)

projs_cache["lotss_initial_pure"] = {
    "projs":     np.concatenate([_X_tr_cp, _X_te_cp]),
    "labels":    np.concatenate([_y_tr_cp, _y_te_cp]),
    "split_ids": np.concatenate([
        np.zeros(len(_X_tr_cp), dtype=np.int64),
        np.full(len(_X_te_cp), 2, dtype=np.int64),
    ]),
}
print(f"Loaded lotss_initial_pure: train={len(_X_tr_cp)}  test={len(_X_te_cp)}  proj_dim={_X_tr_cp.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '../outputs/byol_runs/enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar2_cov0.1_gamma0.25_f1_sw0.02_augextended_20260707_0934/data/train_labels.npy'

## 4. Classifier Utilities

Helper functions used by Section 5. `run_linear_probe` fits a logistic regression
(`max_iter=1000`, `C=1.0`) on train-split projections normalised with `StandardScaler`.
`run_classifier_suite` trains KNN, Random Forest, and a small MLP.
Both are called on-site as a fallback when pre-computed JSON results are not found.

In [ ]:
def run_linear_probe(cache_entry, n_classes):
    """Fit logistic regression on train projections, report val & test metrics.

    Val is optional — skipped when split_ids contains no 1s (e.g. BYOL split).
    Returns dict with keys "test" (and "val" if present), plus "clf" and "scaler".
    """
    projs     = cache_entry["projs"]
    labels    = cache_entry["labels"]
    split_ids = cache_entry["split_ids"]

    X = {s: projs[split_ids == i]  for s, i in [("train",0),("val",1),("test",2)]}
    y = {s: labels[split_ids == i] for s, i in [("train",0),("val",1),("test",2)]}

    scaler     = StandardScaler()
    X["train"] = scaler.fit_transform(X["train"])
    if len(X["val"]):
        X["val"] = scaler.transform(X["val"])
    X["test"]  = scaler.transform(X["test"])

    clf = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf.fit(X["train"], y["train"])

    results = {}
    for split in ("val", "test"):
        if len(X[split]) == 0:
            continue
        y_pred = clf.predict(X[split])
        y_prob = clf.predict_proba(X[split])
        acc    = accuracy_score(y[split], y_pred)
        f1     = f1_score(y[split], y_pred, average="macro", zero_division=0)
        rec    = recall_score(y[split], y_pred, average="macro", zero_division=0)
        if n_classes == 2:
            auc = roc_auc_score(y[split], y_prob[:, 1])
        else:
            auc = roc_auc_score(
                label_binarize(y[split], classes=list(range(n_classes))),
                y_prob, multi_class="ovr", average="macro",
            )
        results[split] = {"accuracy": acc, "f1": f1, "recall": rec, "auc": auc, "n": len(y[split])}

    results["clf"]    = clf
    results["scaler"] = scaler
    return results


## 5. Classifier Suite

Loads pre-computed classifier results (LR / KNN / RF) from `data/classifiers/*.json` if
available. Falls back to training on site via `run_linear_probe` and `run_classifier_suite`
for any missing classifiers (LR, KNN, RF, MLP). The MLP has no pre-computed JSON and is
always trained here.

Also loads multirun RF statistics from `run_summary.json` (mean ± std and top-5 ensemble)
when available.

In [ ]:
# ── MLP ───────────────────────────────────────────────────────────────────────
class _MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(512, 256), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(prev, n_classes)

    def forward(self, x):
        return self.head(self.body(x))


def run_classifier_suite(cache_entry, n_classes,
                         n_neighbors=15, n_estimators=200,
                         mlp_hidden=(512, 256), mlp_epochs=100,
                         mlp_patience=15, mlp_lr=1e-3, mlp_batch=256,
                         mlp_val_frac=0.15):
    """
    Train KNN, Random Forest, and MLP on train-split BYOL projections.
    When no val split exists (BYOL split), carves mlp_val_frac from train for MLP early stopping.
    Returns dict keyed by classifier name, each with accuracy/f1/recall/auc/n.
    """
    projs     = cache_entry["projs"]
    labels    = cache_entry["labels"]
    split_ids = cache_entry["split_ids"]

    X = {s: projs[split_ids == i]  for s, i in [("train", 0), ("val", 1), ("test", 2)]}
    y = {s: labels[split_ids == i] for s, i in [("train", 0), ("val", 1), ("test", 2)]}

    scaler     = StandardScaler()
    X["train"] = scaler.fit_transform(X["train"])
    if len(X["val"]):
        X["val"] = scaler.transform(X["val"])
    X["test"]  = scaler.transform(X["test"])

    def _metrics(y_true, y_pred, y_prob):
        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
        if n_classes == 2:
            auc = roc_auc_score(y_true, y_prob[:, 1])
        else:
            auc = roc_auc_score(
                label_binarize(y_true, classes=list(range(n_classes))),
                y_prob, multi_class="ovr", average="macro",
            )
        return {"accuracy": acc, "f1": f1, "recall": rec, "auc": auc, "n": len(y_true)}

    results = {}

    # ── KNN ──────────────────────────────────────────────────────────────────
    print("  KNN...", end=" ", flush=True)
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="euclidean", n_jobs=-1)
    knn.fit(X["train"], y["train"])
    y_pred = knn.predict(X["test"])
    y_prob = knn.predict_proba(X["test"])
    results["knn"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['knn']['f1']:.3f}")

    # ── Random Forest ─────────────────────────────────────────────────────────
    print("  Random Forest...", end=" ", flush=True)
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=SEED, n_jobs=-1)
    rf.fit(X["train"], y["train"])
    y_pred = rf.predict(X["test"])
    y_prob = rf.predict_proba(X["test"])
    results["random_forest"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['random_forest']['f1']:.3f}")

    # ── MLP ───────────────────────────────────────────────────────────────────
    print("  MLP...", end=" ", flush=True)

    # When no val split exists, carve a fraction from train for early stopping
    if len(X["val"]) == 0:
        rng_v   = np.random.default_rng(SEED)
        n_val   = max(1, int(len(X["train"]) * mlp_val_frac))
        val_idx = rng_v.choice(len(X["train"]), size=n_val, replace=False)
        tr_idx  = np.setdiff1d(np.arange(len(X["train"])), val_idx)
        X_tr_mlp = X["train"][tr_idx];  y_tr_mlp = y["train"][tr_idx]
        X_va_mlp = X["train"][val_idx]; y_va_mlp = y["train"][val_idx]
    else:
        X_tr_mlp, y_tr_mlp = X["train"], y["train"]
        X_va_mlp, y_va_mlp = X["val"],   y["val"]

    X_tr_t = torch.from_numpy(X_tr_mlp).float().to(device)
    y_tr_t = torch.from_numpy(y_tr_mlp).long().to(device)
    X_va_t = torch.from_numpy(X_va_mlp).float().to(device)
    y_va_t = torch.from_numpy(y_va_mlp).long().to(device)

    mlp = _MLP(X_tr_t.shape[1], n_classes, hidden=mlp_hidden).to(device)
    opt = torch.optim.Adam(mlp.parameters(), lr=mlp_lr, weight_decay=1e-4)
    rng = np.random.default_rng(SEED)

    best_val, best_state, wait = float("inf"), None, 0
    for epoch in range(mlp_epochs):
        mlp.train()
        perm = rng.permutation(len(X_tr_t))
        for i in range(0, len(X_tr_t), mlp_batch):
            idx = perm[i:i + mlp_batch]
            loss = nn.CrossEntropyLoss()(mlp(X_tr_t[idx]), y_tr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()

        mlp.eval()
        with torch.no_grad():
            val_loss = nn.CrossEntropyLoss()(mlp(X_va_t), y_va_t).item()
        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
            wait       = 0
        else:
            wait += 1
            if wait >= mlp_patience:
                break

    mlp.load_state_dict(best_state)
    mlp.eval()
    with torch.no_grad():
        X_te_t = torch.from_numpy(X["test"]).float().to(device)
        y_prob  = torch.softmax(mlp(X_te_t), dim=1).cpu().numpy()
        y_pred  = y_prob.argmax(axis=1)
    results["mlp"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['mlp']['f1']:.3f}")

    return results


In [ ]:
_run_dir = Path(CONFIG["checkpoint"]).parent
_clf_dir = _run_dir / "data" / "classifiers"
_feat    = "projections"
_ls_key  = "initial_pure"

suite_results = {"lotss_initial_pure": {}}

# ── Load pre-computed results ─────────────────────────────────────────────────
for _clf_name, _suite_key in [("rf", "random_forest"), ("knn", "knn"), ("lr", "lr")]:
    _json_path = _clf_dir / f"{_clf_name}_{_ls_key}_{_feat}.json"
    if _json_path.exists():
        with open(_json_path) as _f:
            _d = json.load(_f)
        suite_results["lotss_initial_pure"][_suite_key] = {
            "f1":       _d["f1_macro"],
            "auc":      _d["auc_macro"],
            "accuracy": _d["accuracy"],
            "recall":   _d["recall_macro"],
            "n":        _d["n_test"],
        }
        print(f"  {_clf_name.upper()}: F1={_d['f1_macro']:.3f}  AUC={_d['auc_macro']:.3f}  Acc={_d['accuracy']:.3f}")
    else:
        print(f"  {_clf_name.upper()}: not found ({_json_path})")

# ── Fallback: train KNN / RF / MLP on site ───────────────────────────────────
_need_suite = [k for k in ("knn", "random_forest") if k not in suite_results["lotss_initial_pure"]]
if _need_suite or "mlp" not in suite_results["lotss_initial_pure"]:
    _all_missing = _need_suite + ([] if "mlp" in suite_results["lotss_initial_pure"] else ["mlp"])
    print(f"  Training on site: {_all_missing}...")
    _trained = run_classifier_suite(projs_cache["lotss_initial_pure"], n_classes=5)
    for _k, _v in _trained.items():
        if _k not in suite_results["lotss_initial_pure"]:
            suite_results["lotss_initial_pure"][_k] = _v

# ── Fallback: linear probe (LR) on site ──────────────────────────────────────
if "lr" not in suite_results["lotss_initial_pure"]:
    print("  Linear probe (LR): training on site...")
    _lp_res = run_linear_probe(projs_cache["lotss_initial_pure"], n_classes=5)
    suite_results["lotss_initial_pure"]["lr"] = _lp_res["test"]

_lr = suite_results["lotss_initial_pure"].get("lr", {})
if _lr:
    print(f"  LR (linear probe): F1={_lr['f1']:.3f}  AUC={_lr['auc']:.3f}  Acc={_lr['accuracy']:.3f}")

# ── Multirun RF: mean/std and top-5 ensemble ──────────────────────────────────
_multirun_dir = _clf_dir / f"multirun_{_ls_key}_{_feat}"
_summary_path = _multirun_dir / "run_summary.json"
multirun_results = {}
if _summary_path.exists():
    with open(_summary_path) as _f:
        _mr = json.load(_f)
    multirun_results["lotss_initial_pure"] = _mr
    rf_mr = _mr.get("rf", {})
    if rf_mr:
        suite_results["lotss_initial_pure"]["random_forest"].update({
            "f1":           rf_mr["f1_macro_mean"],
            "auc":          rf_mr["auc_macro_mean"],
            "accuracy":     rf_mr["accuracy_mean"],
            "recall":       rf_mr["recall_macro_mean"],
            "f1_std":       rf_mr["f1_macro_std"],
            "auc_std":      rf_mr["auc_macro_std"],
            "accuracy_std": rf_mr["accuracy_std"],
            "recall_std":   rf_mr["recall_macro_std"],
        })
        _top5 = rf_mr["ensemble"]["top-5"]
        suite_results["lotss_initial_pure"]["rf_top5"] = {
            "f1":       _top5["f1_macro"],
            "auc":      _top5["auc_macro"],
            "accuracy": _top5["accuracy"],
            "recall":   _top5["recall_macro"],
        }
        print(f"  RF ({_mr['n_runs']} runs): "
              f"F1={rf_mr['f1_macro_mean']:.3f}±{rf_mr['f1_macro_std']:.3f}  "
              f"AUC={rf_mr['auc_macro_mean']:.3f}±{rf_mr['auc_macro_std']:.3f}")
        print(f"  RF top-5 ensemble: F1={_top5['f1_macro']:.3f}  AUC={_top5['auc_macro']:.3f}")


## 6. Supervised Baseline Comparison

Load pre-trained supervised classifier results from `CONFIG["baselines_dir"]`
and compare against all BYOL classifiers on the LoTSS initial_pure test split.


In [ ]:
_KNOWN_MODELS = ("enb0", "dualssn", "vit", "cnn", "scatternet", "simplescatternet")

_baselines_dir = Path(CONFIG.get("baselines_dir", "../outputs/supervised_baseline_classifiers"))
baseline_results = {}

if _baselines_dir.exists():
    # Group run dirs by model type
    _model_runs = {}
    for _path in sorted(_baselines_dir.rglob("results.json")):
        with open(_path) as _f:
            _res = json.load(_f)
        # Accept any 5-class initial label set (class name strings vary between runs)
        if len(_res.get("class_names", [])) != 5:
            continue
        _dir   = _path.parent.name
        _model = next((m for m in _KNOWN_MODELS if m in _dir.lower()), _dir)
        _ls    = _res.get("label_set", "?")
        _key   = f"{_model} ({_ls})" if _ls != "?" else _model
        _model_runs.setdefault(_key, []).append((_path.parent, _res))

    for _key, _runs in _model_runs.items():
        _metrics_list = [r for _, r in _runs]
        for _m in ("f1_macro", "auc_macro", "accuracy", "recall_macro"):
            _vals = [r[_m] for r in _metrics_list if _m in r]
            baseline_results.setdefault(_key, {})[_m + "_mean"] = float(np.mean(_vals))
            baseline_results[_key][_m + "_std"] = float(np.std(_vals)) if len(_vals) > 1 else None

        # Top-5 ensemble: sort by f1_macro, average probs from top-5 runs
        _sorted_runs = sorted(_runs, key=lambda x: x[1].get("f1_macro", 0), reverse=True)
        _top_k = _sorted_runs[:min(5, len(_sorted_runs))]
        _prob_list, _y_true = [], None
        for _rd, _ in _top_k:
            _p = _rd / "test_probs.npy"
            _l = _rd / "test_labels.npy"
            if _p.exists() and _l.exists():
                _prob_list.append(np.load(_p))
                if _y_true is None:
                    _y_true = np.load(_l)
        if _prob_list and _y_true is not None:
            _n_cls     = _prob_list[0].shape[1]
            _avg_probs = np.mean(_prob_list, axis=0)
            _y_int     = _y_true.argmax(axis=1) if _y_true.ndim == 2 else _y_true
            _pure      = (_y_true.sum(axis=1) == 1) if _y_true.ndim == 2 else np.ones(len(_y_true), bool)
            _avg_probs = _avg_probs[_pure]
            _y_pred    = _avg_probs.argmax(axis=1)
            _y_int     = _y_int[_pure]
            baseline_results[_key]["top5"] = {
                "f1_macro":     f1_score(_y_int, _y_pred, average="macro", zero_division=0),
                "auc_macro":    roc_auc_score(label_binarize(_y_int, classes=list(range(_n_cls))),
                                              _avg_probs, multi_class="ovr", average="macro"),
                "accuracy":     accuracy_score(_y_int, _y_pred),
                "recall_macro": recall_score(_y_int, _y_pred, average="macro", zero_division=0),
            }
        _n = len(_runs)
        _f1m = baseline_results[_key]["f1_macro_mean"]
        _f1s_v = baseline_results[_key]["f1_macro_std"] or 0
        print(f"  {_key}: {_n} run(s)  F1={_f1m:.3f}±{_f1s_v:.3f}")
    print(f"Loaded {len(baseline_results)} baseline group(s): {list(baseline_results.keys())}")
else:
    print(f"Baselines directory not found: {_baselines_dir}")

# ── Build combined table (string-formatted to support mean ± std display) ──────
def _fmt(r, key, std_key=None):
    v = r.get(key, float("nan"))
    if v != v:
        return "—"
    s = r.get(std_key)
    return f"{v:.3f} ± {s:.3f}" if s is not None else f"{v:.3f}"

_suite = suite_results.get("lotss_initial_pure", {})

_byol_rows = [
    ("BYOL — Logistic Regression", _suite.get("lr",            {}), False),
    ("BYOL — KNN",                 _suite.get("knn",           {}), False),
    ("BYOL — Random Forest",       _suite.get("random_forest", {}), True),
    ("BYOL — RF top-5 ensemble",   _suite.get("rf_top5",       {}), False),
    ("BYOL — MLP",                 _suite.get("mlp",           {}), False),
]

_rows = []
for _name, _r, _use_std in _byol_rows:
    if not _r:
        continue
    _rows.append({
        "Method":       _name,
        "Accuracy":     _fmt(_r, "accuracy", "accuracy_std" if _use_std else None),
        "Macro F1":     _fmt(_r, "f1",       "f1_std"       if _use_std else None),
        "Macro Recall": _fmt(_r, "recall",   "recall_std"   if _use_std else None),
        "Macro AUC":    _fmt(_r, "auc",      "auc_std"      if _use_std else None),
    })

for _name, _br in baseline_results.items():
    _rows.append({
        "Method":       f"Supervised — {_name.upper()}",
        "Accuracy":     _fmt(_br, "accuracy_mean",     "accuracy_std"),
        "Macro F1":     _fmt(_br, "f1_macro_mean",     "f1_macro_std"),
        "Macro Recall": _fmt(_br, "recall_macro_mean", "recall_macro_std"),
        "Macro AUC":    _fmt(_br, "auc_macro_mean",    "auc_macro_std"),
    })
    _top5 = _br.get("top5")
    if _top5:
        _rows.append({
            "Method":       f"Supervised — {_name.upper()} top-5 ensemble",
            "Accuracy":     f"{_top5['accuracy']:.3f}",
            "Macro F1":     f"{_top5['f1_macro']:.3f}",
            "Macro Recall": f"{_top5['recall_macro']:.3f}",
            "Macro AUC":    f"{_top5['auc_macro']:.3f}",
        })

df_vs_baselines = pd.DataFrame(_rows).set_index("Method")
print("\n5-class Initial Classification — BYOL vs Supervised Baselines (LoTSS test split)")
display(df_vs_baselines)

## 6b. Finetuning Comparison

Per-class and macro F1 metrics for different finetuning strategies (freeze0, freeze4, supervised1)
across BYOL runs. Reads `finetuning_metrics.json` from each run's finetuning subdirectory.

In [ ]:
model_paths    = CONFIG["finetuning_runs"]
finetune_types = CONFIG["finetuning_types"]

per_class_metrics = ["accuracy", "precision", "recall", "f1", "auc"]

def _nested_get(obj, path):
    cur = obj
    for key in path:
        if isinstance(cur, dict) and key in cur:
            cur = cur[key]
        else:
            return None
    return cur


def _extract_per_class_metric(metrics, metric_name):
    candidates = [
        (metric_name, "test"),
        (metric_name,),
        ("per_class", metric_name),
        ("test", metric_name),
        ("classification", metric_name, "test"),
    ]
    value = None
    for path in candidates:
        value = _nested_get(metrics, path)
        if value is not None:
            break

    if value is None:
        return None

    if isinstance(value, dict):
        for k in ("per_class", "values", "by_class", "test"):
            if k in value:
                value = value[k]
                break

    arr = np.asarray(value, dtype=np.float32)
    if arr.ndim == 0:
        return None
    return arr.reshape(-1)


def _extract_macro_f1(metrics):
    candidates = [
        ("macro_f1", "test"),
        ("macro_f1",),
        ("f1_macro", "test"),
        ("f1_macro",),
        ("macro", "f1"),
    ]
    value = None
    for path in candidates:
        value = _nested_get(metrics, path)
        if value is not None:
            break

    if value is None:
        return None

    if isinstance(value, dict):
        for k in ("value", "score", "test"):
            if k in value:
                value = value[k]
                break

    arr = np.asarray(value, dtype=np.float32)
    if arr.ndim == 0:
        return float(arr)
    if arr.size == 0:
        return None
    return float(arr.mean())


metrics_by_type = {metric: {ftype: {} for ftype in finetune_types} for metric in per_class_metrics}
macro_f1_results = {ftype: {} for ftype in finetune_types}

for ftype in finetune_types:
    for model_path in model_paths:
        metrics_path = Path(model_path) / ftype / "finetuning_metrics.json"
        if not os.path.exists(metrics_path):
            continue
        with open(metrics_path) as _f:
            _metrics = json.load(_f)
        for metric_name in per_class_metrics:
            per_class_vals = _extract_per_class_metric(_metrics, metric_name)
            if per_class_vals is not None:
                metrics_by_type[metric_name][ftype][model_path] = per_class_vals
        macro_val = _extract_macro_f1(_metrics)
        if macro_val is not None:
            macro_f1_results[ftype][model_path] = macro_val

for metric_name in per_class_metrics:
    plot_model_paths = [
        mp for mp in model_paths
        if all(mp in metrics_by_type[metric_name][ftype] for ftype in finetune_types)
    ]
    if not plot_model_paths:
        print(f"Skipping {metric_name}: no complete set across finetuning types.")
        continue

    num_classes = len(metrics_by_type[metric_name][finetune_types[0]][plot_model_paths[0]])
    class_idx = np.arange(num_classes)
    ncols = min(3, len(plot_model_paths))
    nrows = int(np.ceil(len(plot_model_paths) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.5 * nrows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, mp in zip(axes, plot_model_paths):
        model_name = Path(mp.rstrip("/")).name.replace("run_run_mlp_", "")
        for ftype in finetune_types:
            ax.plot(class_idx, metrics_by_type[metric_name][ftype][mp],
                    marker="o", linewidth=1.8, label=ftype.replace("finetuning_", ""))
        ax.set_title(model_name)
        ax.set_xticks(class_idx)
        ax.set_xlabel("Class")
        ax.grid(True, alpha=0.3)

    for ax in axes[len(plot_model_paths):]:
        ax.axis("off")

    axes[0].set_ylabel(f"Test {metric_name}")
    axes[0].legend(title="Finetuning")
    fig.suptitle(f"Per-class test {metric_name}: freeze0 vs freeze4 vs supervised1", y=1.02)
    fig.tight_layout()
    plt.show()

# ── Macro F1 bar chart ────────────────────────────────────────────────────────
macro_plot_model_paths = [
    mp for mp in model_paths
    if all(mp in macro_f1_results[ftype] for ftype in finetune_types)
]

if not macro_plot_model_paths:
    print("Skipping macro_f1 plot: no complete set across finetuning types.")
else:
    x = np.arange(len(macro_plot_model_paths))
    width = 0.8 / len(finetune_types)

    plt.figure(figsize=(max(8, 1.8 * len(macro_plot_model_paths)), 4.5))
    for i, ftype in enumerate(finetune_types):
        values = [macro_f1_results[ftype][mp] for mp in macro_plot_model_paths]
        offset = (i - (len(finetune_types) - 1) / 2) * width
        plt.bar(x + offset, values, width=width, label=ftype.replace("finetuning_", ""))

    plt.xticks(x, [Path(mp.rstrip("/")).name.replace("run_run_mlp_", "") for mp in macro_plot_model_paths],
               rotation=25, ha="right")
    plt.ylabel("Macro F1 (test)")
    plt.title("Macro F1 across models")
    plt.grid(axis="y", alpha=0.3)
    plt.legend(title="Finetuning")
    plt.tight_layout()
    plt.show()


## 6c. F1 Comparison: BYOL vs Supervised

Scans all BYOL runs in `byol_runs_dir` for classifier results and compares against
supervised baselines. Reads configs from each run's `logs/configuration_log.txt`.

- **BYOL + LR** — logistic regression on frozen features (`run_summary.json`)
- **BYOL + FT** — fine-tuned encoder (`finetuning_metrics.json`, type set in `CONFIG`)
- **Supervised** — end-to-end supervised classifier

X-axis stores `n_labels` as a real number per config (rendered categorically now;
switch to numeric later by changing the `ax.set_xticks` block).

In [ ]:
_ls_key   = CONFIG.get("label_set", "initial_pure")
_runs_dir = Path(CONFIG["byol_runs_dir"])
_ft_runs  = [Path(p) for p in CONFIG["finetuning_runs"]]
_ft_type  = CONFIG["finetuning_type"]
_base_dir = Path(CONFIG["baselines_dir"])

_KNOWN_MODELS_6C = ("enb0", "dualssn", "vit", "cnn", "scatternet", "simplescatternet")
_SPREAD_THRESH   = 1e-3   # std below this → treat as deterministic, skip violin

# ── Helpers ────────────────────────────────────────────────────────────────────
def _parse_cfg_log(run_dir):
    """Return dict from configuration_log.txt (key: value lines)."""
    p = Path(run_dir) / "logs" / "configuration_log.txt"
    if not p.exists():
        return {}
    cfg = {}
    for line in p.read_text().splitlines():
        if ": " in line:
            k, v = line.split(": ", 1)
            cfg[k.strip()] = v.strip()
    return cfg

def _macro_f1_from_dict(d):
    """Extract macro F1 scalar from a metrics dict with flexible key layout."""
    for path in [("f1_macro",), ("macro_f1",), ("f1_macro", "test"), ("macro_f1", "test"),
                 ("macro", "f1")]:
        v = d
        for k in path:
            v = v.get(k) if isinstance(v, dict) else None
            if v is None:
                break
        if v is not None:
            if isinstance(v, dict):
                v = next((v[k] for k in ("value", "score", "test") if k in v), next(iter(v.values()), None))
            try:
                arr = np.asarray(v, dtype=float)
                return float(arr.mean()) if arr.size > 0 else None
            except Exception:
                pass
    return None

def _read_n_labels(run_dir, feat):
    """N labelled training samples used for classifier (initial_pure split)."""
    for clf in ("lr", "rf", "knn"):
        p = Path(run_dir) / "data" / "classifiers" / f"{clf}_{_ls_key}_{feat}.json"
        if p.exists():
            d = json.load(open(p))
            if "n_train" in d:
                return int(d["n_train"])
    # Fallback: read projection file shape (approximation, not filtered to initial_pure)
    proj = Path(run_dir) / "data" / "byol" / "labelled_train_projections.npy"
    if proj.exists():
        return int(np.load(proj, mmap_mode="r").shape[0])
    return None

def _read_individual_f1s(multirun_dir, clf_key):
    """Try to load per-seed F1 values from individual JSON files in multirun_dir."""
    vals = []
    for p in sorted(Path(multirun_dir).glob("run_*.json")):
        try:
            d = json.load(open(p))
            sub = d.get(clf_key, d)
            v = _macro_f1_from_dict(sub)
            if v is not None:
                vals.append(v)
        except Exception:
            pass
    return vals or None

# ── 1. BYOL + LR ──────────────────────────────────────────────────────────────
print("── BYOL + LR ────────────────────────────────────────────────────────────")
_byol_lr = []

for _rd in sorted(_runs_dir.iterdir()):
    if not _rd.is_dir():
        continue
    _cfg = _parse_cfg_log(_rd)
    if not _cfg:
        continue
    sw      = float(_cfg.get("supervision_weight", "nan"))
    f_label = float(_cfg.get("f_label", "nan"))

    # Find whichever feature space has a multirun summary
    feat, summary, mrun_dir = None, None, None
    for _f in ("projections", "encodings"):
        _mdir = _rd / "data" / "classifiers" / f"multirun_{_ls_key}_{_f}"
        _sp   = _mdir / "run_summary.json"
        if _sp.exists():
            summary  = json.load(open(_sp))
            feat     = _f
            mrun_dir = _mdir
            break
    if summary is None:
        continue

    lr_agg = summary.get("lr", {})
    if not lr_agg:
        continue

    n_runs   = summary.get("n_runs", 1)
    f1_mean  = float(lr_agg.get("f1_macro_mean", float("nan")))
    f1_std   = float(lr_agg.get("f1_macro_std",  0.0))
    f1_vals  = _read_individual_f1s(mrun_dir, "lr")
    n_labels = _read_n_labels(_rd, feat)

    print(f"  series=BYOL+LR  sw={sw}  f_label={f_label}  feat={feat}  n_runs={n_runs}  F1={f1_mean:.3f}")
    _byol_lr.append(dict(sw=sw, f_label=f_label, feat=feat, n_runs=n_runs,
                         f1_mean=f1_mean, f1_std=f1_std, f1_vals=f1_vals,
                         n_labels=n_labels))

# ── 2. BYOL + Fine-tuning ─────────────────────────────────────────────────────
print(f"\n── BYOL + Fine-tuning ({_ft_type}) ──────────────────────────────────────")
_byol_ft = []

for _rd in _ft_runs:
    if not _rd.exists():
        print(f"  WARNING: not found: {_rd}")
        continue
    _cfg = _parse_cfg_log(_rd)
    sw      = float(_cfg.get("supervision_weight", "nan")) if _cfg else float("nan")
    f_label = float(_cfg.get("f_label",            "nan")) if _cfg else float("nan")

    _mp = _rd / _ft_type / "finetuning_metrics.json"
    if not _mp.exists():
        print(f"  WARNING: {_mp.name} not found in {_rd.name}")
        continue
    f1 = _macro_f1_from_dict(json.load(open(_mp)))
    if f1 is None:
        print(f"  WARNING: could not extract macro F1 from {_mp}")
        continue

    n_labels = _read_n_labels(_rd, "projections")
    print(f"  series=BYOL+FT  sw={sw}  f_label={f_label}  feat={_ft_type}  n_runs=1  F1={f1:.3f}")
    _byol_ft.append(dict(sw=sw, f_label=f_label, feat=_ft_type, n_runs=1,
                         f1_mean=f1, f1_std=0.0, f1_vals=[f1],
                         n_labels=n_labels))

# ── 3. Supervised baselines ───────────────────────────────────────────────────
print("\n── Supervised Baselines ─────────────────────────────────────────────────")
_sup = []

if _base_dir.exists():
    _model_grps = {}
    for _rjson in sorted(_base_dir.rglob("results.json")):
        _res = json.load(open(_rjson))
        if len(_res.get("class_names", [])) != 5:
            continue
        _model = next((m for m in _KNOWN_MODELS_6C if m in _rjson.parent.name.lower()),
                      _rjson.parent.name)
        _model_grps.setdefault(_model, []).append(_res)
    for _model, _runs in _model_grps.items():
        _vals = [r["f1_macro"] for r in _runs if "f1_macro" in r]
        if not _vals:
            continue
        f1_mean = float(np.mean(_vals))
        f1_std  = float(np.std(_vals)) if len(_vals) > 1 else 0.0
        print(f"  series=Supervised  model={_model}  n_runs={len(_vals)}  F1={f1_mean:.3f}±{f1_std:.3f}")
        _sup.append(dict(model=_model, n_runs=len(_vals), f1_mean=f1_mean, f1_std=f1_std,
                         f1_vals=_vals, n_labels=None))
else:
    print(f"  not found: {_base_dir}")

# ── Style mapping ──────────────────────────────────────────────────────────────
# Line styles activate when x becomes numeric; wired in now.
_LINESTYLES = {"byol_lr": "-",  "byol_ft": "-.", "baseline": "--"}
_MARKERS    = {"byol_lr": "o",  "byol_ft": "s",  "baseline": "^"}

_all_sw    = sorted({r["sw"] for r in _byol_lr + _byol_ft if not np.isnan(r["sw"])})
_sw_colors = {sw: plt.cm.tab10(i % 10) for i, sw in enumerate(_all_sw)}

# ── Build plot rows ────────────────────────────────────────────────────────────
# Each row: (series_key, display_label, color, n_labels [real], f1_mean, f1_std, f1_vals)
_plot_rows = []

for r in _byol_lr:
    color = _sw_colors.get(r["sw"], "grey")
    label = f"BYOL + LR (sw={r['sw']}, f={r['f_label']}, {r['feat']})"
    _plot_rows.append(("byol_lr", label, color, r["n_labels"],
                       r["f1_mean"], r["f1_std"], r["f1_vals"]))

for r in _byol_ft:
    color = _sw_colors.get(r["sw"], "grey")
    label = f"BYOL + FT (sw={r['sw']}, f={r['f_label']})"
    _plot_rows.append(("byol_ft", label, color, r["n_labels"],
                       r["f1_mean"], r["f1_std"], r["f1_vals"]))

for r in _sup:
    _plot_rows.append(("baseline", f"Supervised — {r['model'].upper()}", "steelblue",
                       r["n_labels"], r["f1_mean"], r["f1_std"], r["f1_vals"]))

# ── Plot ───────────────────────────────────────────────────────────────────────
if not _plot_rows:
    print("\nNo data to plot.")
else:
    fig, ax = plt.subplots(figsize=(max(7, len(_plot_rows) * 1.4), 5))
    _legend_handles = {}

    for i, (series, label, color, n_labels, f1_mean, f1_std, f1_vals) in enumerate(_plot_rows):
        spread     = np.std(f1_vals) if f1_vals and len(f1_vals) > 1 else (f1_std or 0.0)
        use_violin = f1_vals is not None and len(f1_vals) > 2 and spread > _SPREAD_THRESH

        if use_violin:
            vp = ax.violinplot(f1_vals, positions=[i], widths=0.5,
                               showmeans=True, showmedians=False, showextrema=True)
            for pc in vp["bodies"]:
                pc.set_facecolor(color); pc.set_alpha(0.45)
            for part in ("cmeans", "cmaxes", "cmins", "cbars"):
                if part in vp:
                    vp[part].set_edgecolor(color); vp[part].set_linewidth(1.5)
            # Proxy handle for legend
            if label not in _legend_handles:
                _legend_handles[label] = mpatches.Patch(facecolor=color, alpha=0.5, label=label)
        else:
            h, = ax.plot([i], [f1_mean], marker=_MARKERS[series], color=color,
                         linestyle="None", markersize=9, label=label)
            if spread > _SPREAD_THRESH:
                ax.errorbar([i], [f1_mean], yerr=f1_std, color=color, capsize=4, zorder=3)
            if label not in _legend_handles:
                _legend_handles[label] = h

    # ── X-axis: categorical now, real n_labels stored per row ─────────────────
    # To switch to numeric later: replace set_xticks/set_xticklabels with
    #   ax.set_xticks([r[3] for r in _plot_rows]); ax.set_xticklabels(...)
    #   and change positions=[i] above to positions=[r[3]]
    _x_labels = [str(int(n)) if n is not None else "?" for _, _, _, n, *_ in _plot_rows]
    ax.set_xticks(range(len(_plot_rows)))
    ax.set_xticklabels(_x_labels, rotation=30, ha="right")
    ax.set_xlabel("N labels used to train classifier")
    ax.set_ylabel("Macro F1")
    ax.set_title(f"Classification F1: BYOL vs Supervised — {_ls_key}")
    ax.set_ylim(bottom=0.0)
    ax.grid(axis="y", alpha=0.3)

    # Series-style legend strip (ready for when lines activate)
    _style_strip = [
        plt.Line2D([0], [0], color="grey", linestyle=_LINESTYLES[s],
                   marker=_MARKERS[s], markersize=7, label=n)
        for s, n in [("byol_lr",  "BYOL + LR"),
                     ("byol_ft",  f"BYOL + FT ({_ft_type})"),
                     ("baseline", "Supervised")]
    ]
    _data_handles = list(_legend_handles.values())
    ax.legend(handles=_data_handles + _style_strip,
              loc="lower right", fontsize=8,
              ncol=max(1, len(_data_handles) // 10 + 1))

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/f1_comparison_{_ls_key}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 7. Hyperparameter Sweep Grid

Reads pre-computed classifier results from `data/classifiers/` across all BYOL run
directories and produces sweep line plots. The reference run (from `CONFIG["checkpoint"]`) pins
all hyperparameters except the two axes (`axis_x` / `axis_y`).

Configure sweep axes, classifier, and run search pattern in `GRID_CONFIG` (Section 0).
Each plot shows metric vs. one sweep variable, with lines coloured by the other variable
and linestyle indicating the downstream classifier (RF / KNN / LR).

In [ ]:
# ── Parseable hyperparameter patterns ─────────────────────────────────────────
_PARAM_RE = {
    "f":         r"_f([\d.]+)_",
    "sw":        r"_sw([\d.]+)_",
    "vicregvar": r"_vicregvar([\d.]+)_",
    "cov":       r"_cov([\d.]+)_",
    "gamma":     r"_gamma([\d.]+)_",
    "ema":       r"_ema([\d.]+)_",
}

def _parse_param(name, param):
    m = re.search(_PARAM_RE[param], name + "_")
    return float(m.group(1)) if m else None

# ── Infer fixed params from the reference run ──────────────────────────────────
_ref_name  = Path(CONFIG["checkpoint"]).parent.name
_axis_x    = GRID_CONFIG["axis_x"]
_axis_y    = GRID_CONFIG["axis_y"]
_fixed = {
    p: _parse_param(_ref_name, p)
    for p in _PARAM_RE
    if p not in (_axis_x, _axis_y) and _parse_param(_ref_name, p) is not None
}
print("Fixed params:", _fixed)

# ── Scan run directories ────────────────────────────────────────────────────────
_root     = Path(GRID_CONFIG["outputs_root"])
_run_dirs = sorted(_root.glob(GRID_CONFIG["run_glob"]))
_run_dirs = [rd for rd in _run_dirs if re.search(r"_f[\d.]+_sw[\d.]+", rd.name)]

_CLFS  = ["rf", "knn", "lr"]
_CLF_STYLES = {"rf": "-", "knn": "--", "lr": ":"}
_CLF_LABELS = {"rf": "RF", "knn": "KNN", "lr": "LR"}
_ls    = GRID_CONFIG["label_set"]
_feat  = GRID_CONFIG["feature_type"]

# Build records per classifier
_records = {}
for _clf in _CLFS:
    _records[_clf] = []
    for rd in _run_dirs:
        params = {p: _parse_param(rd.name, p) for p in _PARAM_RE}
        if not all(
            params.get(k) is not None and abs(params[k] - v) < 1e-9
            for k, v in _fixed.items()
        ):
            continue
        json_path = rd / "data" / "classifiers" / f"{_clf}_{_ls}_{_feat}.json"
        if not json_path.exists():
            continue
        with open(json_path) as _fh:
            _d = json.load(_fh)
        _records[_clf].append({
            "x":   params[_axis_x],
            "y":   params[_axis_y],
            "f1":  _d.get("f1_macro",     float("nan")),
            "auc": _d.get("auc_macro",    float("nan")),
            "acc": _d.get("accuracy",     float("nan")),
            "rec": _d.get("recall_macro", float("nan")),
        })
    print(f"  {_clf}: {len(_records[_clf])} run(s) found")

_grids = {_clf: {(r["x"], r["y"]): r for r in recs}
          for _clf, recs in _records.items()}
_all_recs = [r for recs in _records.values() for r in recs]
_xs = sorted(set(r["x"] for r in _all_recs))
_ys = sorted(set(r["y"] for r in _all_recs))

_METRICS   = [("f1", "F1 macro"), ("auc", "AUC macro"),
              ("acc", "Accuracy"), ("rec", "Recall macro")]
_line_cmap = plt.cm.tab10

# ── Plot function ──────────────────────────────────────────────────────────────
def _plot_sweeps(show_isolated=True):
    for sweep_var, group_var, sweep_vals, group_vals, axis_is_x in [
        (_axis_x, _axis_y, _xs, _ys, True),
        (_axis_y, _axis_x, _ys, _xs, False),
    ]:
        if len(sweep_vals) < 2:
            print(f"Skipping {sweep_var} sweep (only {len(sweep_vals)} value(s))")
            continue

        print(f"Sweep: {sweep_var}  |  {_ls}, {_feat}  |  fixed: {_fixed}")

        fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)

        for ax, (mkey, mlabel) in zip(axes.flat, _METRICS):
            for gi, gv in enumerate(group_vals):
                color = _line_cmap(gi % 10)
                for _clf in _CLFS:
                    xs_plot, ys_plot = [], []
                    for sv in sweep_vals:
                        key = (sv, gv) if axis_is_x else (gv, sv)
                        rec = _grids[_clf].get(key)
                        val = rec[mkey] if rec is not None else float("nan")
                        if val == val:
                            xs_plot.append(sv)
                            ys_plot.append(val)
                    if not xs_plot:
                        continue
                    if not show_isolated and len(xs_plot) == 1:
                        continue
                    ax.plot(xs_plot, ys_plot,
                            marker="o", markersize=6, linewidth=2.0,
                            linestyle=_CLF_STYLES[_clf],
                            color=color,
                            label=f"{group_var}={gv} / {_CLF_LABELS[_clf]}")

            ax.set_xlabel(sweep_var, fontsize=14)
            ax.set_ylabel(mlabel, fontsize=14)
            ax.set_title(mlabel, fontsize=14)
            ax.tick_params(axis="both", labelsize=13)
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)

        _color_handles = [
            plt.Line2D([0], [0], color=_line_cmap(gi % 10), linewidth=2, label=f"{group_var}={gv}")
            for gi, gv in enumerate(group_vals)
        ]
        _style_handles = [
            plt.Line2D([0], [0], color="grey", linewidth=2,
                       linestyle=_CLF_STYLES[c], label=_CLF_LABELS[c])
            for c in _CLFS
        ]
        fig.legend(
            handles=_color_handles + _style_handles,
            loc="upper center", ncol=len(group_vals) + len(_CLFS),
            fontsize=12, bbox_to_anchor=(0.5, 0),
            frameon=True,
        )

        plt.tight_layout()
        _out = f"{OUT_DIR}/sweep_lines_{_ref_name}_{sweep_var}.png"
        plt.savefig(_out, dpi=150, bbox_inches="tight")
        print(f"Saved → {_out}")
        plt.show()

_toggle = widgets.ToggleButton(
    value=False,
    description="Show isolated dots",
    button_style="",
    icon="circle",
    layout=widgets.Layout(width="200px"),
)
widgets.interact(_plot_sweeps, show_isolated=_toggle)

## 8. Confusion Matrix (BYOL vs Supervised Baseline)

Side-by-side row-normalised confusion matrices for a chosen BYOL classifier and a supervised
baseline. Each cell is split: left half (orange) = BYOL, right half (blue) = supervised.

Configure `byol_classifier` and `baseline_model` in `CM_CONFIG` (Section 0).

In [ ]:
_ls  = CM_CONFIG["label_set"]
_key = f"lotss_{_ls}"
_cp  = projs_cache[_key]

y_te_byol = _cp["labels"][_cp["split_ids"] == 2]

_byol_clf  = CM_CONFIG["byol_classifier"]
_run_dir   = Path(CONFIG["checkpoint"]).parent
_clf_dir   = _run_dir / "data" / "classifiers"
_clf_short = {"random_forest": "rf", "knn": "knn", "linear_probe": "lr"}.get(_byol_clf, _byol_clf)
_preds_path = _clf_dir / f"{_clf_short}_{_ls}_projections_test_preds.npy"

if _preds_path.exists():
    y_pred_byol = np.load(_preds_path)
else:
    raise FileNotFoundError(
        f"No pre-computed predictions at {_preds_path}.\n"
        f"Re-run train_byol_classifiers.py with --force on this run to generate them."
    )

print(f"BYOL {_byol_clf}: N_test={len(y_te_byol)}")


BYOL knn: N_test=2005


In [ ]:
_bdir = Path(CM_CONFIG["baseline_dir"])
_tag  = CM_CONFIG["baseline_model"]
n_cls = len([ALL_CLASS_NAMES[i] for i in LABEL_SETS[_ls]])

_candidates = [d for d in _bdir.iterdir()
               if d.is_dir() and _tag in d.name.lower()
               and (d / "test_probs.npy").exists()]
if not _candidates:
    raise FileNotFoundError(f"No baseline run found for '{_tag}' in {_bdir}")
_brun = sorted(_candidates, key=lambda d: d.stat().st_mtime)[-1]

_probs_raw  = np.load(_brun / "test_probs.npy")   # (N, n_classes)
_labels_raw = np.load(_brun / "test_labels.npy")  # (N, n_classes) multi-hot

if _probs_raw.shape[1] != n_cls:
    raise ValueError(
        f"Baseline '{_brun.name}' has {_probs_raw.shape[1]} output classes "
        f"but confusion matrix expects {n_cls}. "
        f"Pick a baseline trained on the same label set."
    )

# Apply initial_pure filter: keep only sources with exactly one initial label
_pure_mask  = _labels_raw.sum(axis=1) == 1
_probs_base = _probs_raw[_pure_mask]
_labels_base = _labels_raw[_pure_mask]

y_pred_base = _probs_base.argmax(axis=1)
y_te_base   = _labels_base.argmax(axis=1)
print(f"Baseline {_tag}: {_brun.name}")
print(f"  N_test (all): {len(_probs_raw)}  →  N_test (pure): {len(_probs_base)}")

Baseline enb0: enb0_initial_pure_byolsplit_enb0_20260622_1720
  N_test (all): 2005  →  N_test (pure): 2005


In [ ]:
_class_names = [ALL_CLASS_NAMES[i] for i in LABEL_SETS[_ls]]
n_cls = len(_class_names)

cm_b = confusion_matrix(y_te_byol,  y_pred_byol, labels=list(range(n_cls)))
cm_s = confusion_matrix(y_te_base,  y_pred_base,  labels=list(range(n_cls)))

# Row-normalise (recall per class)
cm_b_n = cm_b / cm_b.sum(axis=1, keepdims=True).clip(min=1)
cm_s_n = cm_s / cm_s.sum(axis=1, keepdims=True).clip(min=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
_byol_json = _clf_dir / f"{_clf_short}_{_ls}_projections.json"
with open(_byol_json) as _f:
    _bj = json.load(_f)
_byol_metrics = {
    "F1":     _bj["f1_macro"],
    "AUC":    _bj["auc_macro"],
    "Acc":    _bj["accuracy"],
    "Recall": _bj["recall_macro"],
}

_base_auc = roc_auc_score(
    label_binarize(y_te_base, classes=list(range(n_cls))),
    _probs_base, multi_class="ovr", average="macro",
)
_base_metrics = {
    "F1":     f1_score(y_te_base, y_pred_base, average="macro", zero_division=0),
    "AUC":    _base_auc,
    "Acc":    accuracy_score(y_te_base, y_pred_base),
    "Recall": recall_score(y_te_base, y_pred_base, average="macro", zero_division=0),
}

print(f"{'Metric':<8}  {'BYOL':>6}  {'Base':>6}")
print("-" * 26)
for k in ["F1", "AUC", "Acc", "Recall"]:
    print(f"{k:<8}  {_byol_metrics[k]:>6.3f}  {_base_metrics[k]:>6.3f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
cmap_b = plt.cm.Oranges
cmap_s = plt.cm.Blues

fig, ax = plt.subplots(figsize=(n_cls * 1.5 + 3, n_cls * 1.5 + 1))
plt.subplots_adjust(left=0.28, right=0.98)
ax.set_xlim(0, n_cls); ax.set_ylim(0, n_cls)
ax.invert_yaxis()

for i in range(n_cls):
    for j in range(n_cls):
        ax.add_patch(plt.Rectangle([j, i],     0.5, 1, color=cmap_b(cm_b_n[i, j])))
        ax.add_patch(plt.Rectangle([j+0.5, i], 0.5, 1, color=cmap_s(cm_s_n[i, j])))
        ax.text(j+0.25, i+0.5, f"{cm_b_n[i,j]:.2f}", ha="center", va="center", fontsize=13)
        ax.text(j+0.75, i+0.5, f"{cm_s_n[i,j]:.2f}", ha="center", va="center", fontsize=13)
        ax.add_patch(plt.Rectangle([j, i], 1, 1, fill=False, edgecolor="grey", lw=0.5))

ax.set_xticks(np.arange(n_cls) + 0.5)
ax.set_xticklabels(_class_names, rotation=45, ha="right", fontsize=16)
ax.set_yticks(np.arange(n_cls) + 0.5)
ax.set_yticklabels(_class_names, fontsize=16)
ax.set_xlabel("Predicted", fontsize=16)
ax.set_ylabel("True", fontsize=16)

# Metrics table — top left of the axes
_metrics_lines = ["       BYOL  Base"]
for k, short in [("F1", "F1"), ("AUC", "AUC"), ("Acc", "Acc"), ("Recall", "Rec")]:
    _metrics_lines.append(f"{short:<4}  {_byol_metrics[k]:.3f} {_base_metrics[k]:.3f}")
fig.text(0.01, 0.95, "\n".join(_metrics_lines),
         va="top", ha="left",
         fontsize=13, family="monospace",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="grey"))

fig.legend(handles=[
    mpatches.Patch(color=plt.cm.Oranges(0.7), label=f"BYOL {CM_CONFIG['byol_classifier']}"),
    mpatches.Patch(color=plt.cm.Blues(0.7),   label=f"Supervised {CM_CONFIG['baseline_model']}"),
], loc="lower left", fontsize=14, bbox_to_anchor=(0.01, 0.0), frameon=True)

_run_name = _run_dir.name
plt.savefig(f"{OUT_DIR}/confusion_matrix_{_run_name}_{_ls}.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Label-Fraction Experiment

Train logistic regression probes on subsampled fractions of the train-split
projections. Tests how performance scales with the amount of labelled data,
evaluated on the fixed test split.

In [ ]:
# ── Find all BYOL runs with f=0 or sw=0 ───────────────────────────────────────
_outputs_root = Path(GRID_CONFIG["outputs_root"])
_all_runs     = sorted(_outputs_root.glob(GRID_CONFIG["run_glob"]))

_target_runs = [
    rd for rd in _all_runs
    if (_parse_param(rd.name, "f")  == 0.0 or
        _parse_param(rd.name, "sw") == 0.0)
    and (rd / "data" / "byol" / "test_projections.npy").exists()
]
print(f"Found {len(_target_runs)} runs (f=0 or sw=0):")
for rd in _target_runs:
    print(f"  {rd.name}")

# ── Helper: load initial_pure projections from a run dir ──────────────────────
def _load_run_projs(run_dir):
    d = run_dir / "data"
    b = d / "byol"
    X_tr_raw = np.load(b / "labelled_train_projections.npy").astype(np.float32)
    X_te_raw = np.load(b / "test_projections.npy").astype(np.float32)
    # derive splits_dir for label files
    _c = torch.load(run_dir / "byol_model_best.pt", map_location="cpu", weights_only=False)
    _s = run_dir.parent / "data_splits" / str(int(_c["config"]["data_seed"]))
    def _lbl(name):
        p = _s / name
        return np.load(p if p.exists() else d / name)
    lab_path = _s / "labelled_train_labels.npy"
    if not lab_path.exists():
        lab_path = d / "labelled_train_labels.npy"
    if lab_path.exists():
        y_tr_raw = np.load(lab_path)
        if len(y_tr_raw) != len(X_tr_raw):
            # labelled_train_labels.npy was written by a different f_label run.
            # Reconstruct by concatenating labelled + unlabelled labels (covers all of
            # train_idx in order, which matches labelled_train_projections.npy for f=1).
            _unlab = _s / "unlabelled_train_labels.npy"
            if not _unlab.exists():
                _unlab = d / "unlabelled_train_labels.npy"
            if _unlab.exists():
                y_tr_raw = np.concatenate([y_tr_raw, np.load(_unlab)])
            if len(y_tr_raw) != len(X_tr_raw):
                raise ValueError(f"Label count {len(y_tr_raw)} != projection count {len(X_tr_raw)} in {run_dir.name}")
    else:
        y_all = _lbl("train_labels.npy")
        idx   = _lbl("labelled_train_idx.npy")
        y_tr_raw = y_all if len(y_all) == len(X_tr_raw) else y_all[idx]
    y_te_raw = _lbl("test_labels.npy")
    # The shared splits dir may have been overwritten by a run with a different
    # test count (e.g. drop_last). Projections always correspond to the first N
    # test labels, so truncate if the label file is longer.
    if len(y_te_raw) != len(X_te_raw):
        if len(y_te_raw) > len(X_te_raw):
            y_te_raw = y_te_raw[:len(X_te_raw)]
        else:
            raise ValueError(f"test_labels count {len(y_te_raw)} < projection count {len(X_te_raw)} in {run_dir.name}")
    def _pure(y): return y[:, 0:5].sum(axis=1) == 1
    tm, em = _pure(y_tr_raw), _pure(y_te_raw)
    return (X_tr_raw[tm], y_tr_raw[tm][:, 0:5].argmax(1).astype(np.int64),
            X_te_raw[em], y_te_raw[em][:, 0:5].argmax(1).astype(np.int64))

# ── Label-fraction experiment ──────────────────────────────────────────────────
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
N_CLS = 5   # always initial_pure (5 classes)

_clf_factories = {
    "LogReg": lambda: LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
    "KNN":    lambda: KNeighborsClassifier(n_neighbors=15),
    "RF":     lambda: RandomForestClassifier(n_estimators=200, random_state=SEED),
}

def _full_proba(clf, X):
    """Return (N, N_CLS) probability matrix, padding zero columns for unseen classes."""
    y_prob = clf.predict_proba(X)
    if y_prob.shape[1] == N_CLS:
        return y_prob
    full = np.zeros((len(X), N_CLS), dtype=np.float64)
    for j, c in enumerate(clf.classes_):
        full[:, int(c)] = y_prob[:, j]
    return full

def _run_label_frac(X_tr, y_tr, X_te, y_te):
    scaler    = StandardScaler().fit(X_tr)
    X_tr_s    = scaler.transform(X_tr)
    X_te_s    = scaler.transform(X_te)
    rng       = np.random.default_rng(SEED)
    results   = {}
    for f in fractions:
        n   = max(4, int(f * len(X_tr_s)))
        idx = rng.choice(len(X_tr_s), size=n, replace=False)
        results[f] = {}
        for clf_name, clf_fn in _clf_factories.items():
            clf    = clf_fn()
            clf.fit(X_tr_s[idx], y_tr[idx])
            y_pred = clf.predict(X_te_s)
            y_prob = _full_proba(clf, X_te_s)
            results[f][clf_name] = {
                "accuracy": accuracy_score(y_te, y_pred),
                "f1":       f1_score(y_te, y_pred, average="macro", zero_division=0),
                "recall":   recall_score(y_te, y_pred, average="macro", zero_division=0),
                "auc":      roc_auc_score(
                                label_binarize(y_te, classes=list(range(N_CLS))),
                                y_prob, multi_class="ovr", average="macro"),
            }
    return results

_CACHE_FILE = "label_fraction_metrics.json"

def _save_metrics(run_dir, results):
    cache_dir = run_dir / "data" / "classifiers"
    cache_dir.mkdir(parents=True, exist_ok=True)
    # JSON requires string keys
    serialisable = {str(f): v for f, v in results.items()}
    with open(cache_dir / _CACHE_FILE, "w") as fp:
        json.dump(serialisable, fp, indent=2)

def _load_metrics(run_dir):
    cache_path = run_dir / "data" / "classifiers" / _CACHE_FILE
    if not cache_path.exists():
        return None
    with open(cache_path) as fp:
        raw = json.load(fp)
    loaded = {float(f): v for f, v in raw.items()}
    # Invalidate cache if it was built with a different set of fractions
    if set(loaded.keys()) != set(fractions):
        return None
    return loaded

all_frac_results = {}   # {run_label: {f: {clf_name: {metric: val}}}}
for rd in _target_runs:
    f_val  = _parse_param(rd.name, "f")
    sw_val = _parse_param(rd.name, "sw")
    label  = f"f={f_val} sw={sw_val}"

    cached = _load_metrics(rd)
    if cached is not None:
        all_frac_results[label] = cached
        print(f"→ {label}  (loaded from cache)")
        continue

    print(f"\n→ {label}  ({rd.name})  — training...")
    X_tr, y_tr, X_te, y_te = _load_run_projs(rd)
    results = _run_label_frac(X_tr, y_tr, X_te, y_te)
    _save_metrics(rd, results)
    all_frac_results[label] = results
    for f in fractions:
        print(f"  frac={f:.2f}  " +
              "  ".join(f"{nm}: acc={results[f][nm]['accuracy']:.3f}"
                        for nm in _clf_factories))

Found 4 runs (f=0 or sw=0):
  enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar5_cov0.05_gamma0.5_f0_sw0.1_20260621_1950
  enb0_mlp_pd256_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260608_1409
  enb0_mlp_pd50_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260608_1411
  enb0_prepca_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260528_1040
→ f=0.0 sw=0.1  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)


In [ ]:
_metrics    = [("accuracy", "Accuracy"), ("f1", "Macro F1"),
               ("recall", "Macro Recall"), ("auc", "Macro AUC (OvR)")]
_clf_styles = {"LogReg": "-", "KNN": "--", "RF": ":"}
_run_labels = list(all_frac_results.keys())
_palette    = plt.cm.tab10(np.linspace(0, 0.9, max(len(_run_labels), 1)))
_run_colors = {lbl: _palette[i] for i, lbl in enumerate(_run_labels)}

xs_pct = [f * 100 for f in fractions]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (metric, ylabel) in zip(axes, _metrics):
    for run_lbl, res in all_frac_results.items():
        color = _run_colors[run_lbl]
        for clf_name, ls in _clf_styles.items():
            ys = [res[f][clf_name][metric] for f in fractions]
            ax.plot(xs_pct, ys, linestyle=ls, color=color, linewidth=1.8,
                    marker="o", markersize=4, label=f"{run_lbl} / {clf_name}")
    ax.set_xlabel("Label fraction (%)", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.set_title(ylabel, fontsize=24)
    ax.set_xticks(xs_pct)
    ax.set_xticklabels([f"{x:.0f}%" for x in xs_pct], fontsize=18)
    ax.tick_params(axis="y", labelsize=20)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3)

# Shared legend: one entry per (run, clf) — deduplicated from first subplot
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc="upper center",
           ncol=len(_clf_styles), fontsize=14,
           bbox_to_anchor=(0.5, 1.01), frameon=True)

plt.suptitle("Label Fraction vs Performance — BYOL runs with f=0 or sw=0",
             fontsize=39, y=1.07)
plt.tight_layout()
_out = f"{OUT_DIR}/label_fraction_{_ref_name}.png"
plt.savefig(_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {_out}")
